In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/alquiler_idealista.csv', sep=';')
df.head()

,id,tipo_inmueble,titulo,municipio,zona,precio_mensual_eur,habitaciones,superficie_m2,planta,exterior,ascensor,garaje_incluido,garaje_opcional,temporada,lujo,vistas_mar,amueblado,observaciones,Unnamed: 18
0,1,Piso,Piso en los Pinares,Santander,Los Castros,850,3,63,1,1,1,0,0,1,0,0,NaN,Curso escolar 2026-27 reformado,NaN
1,2,Piso,Piso Calleja Norte,Santander,El Sardinero,1400,3,107,3,1,1,1,0,0,0,1,NaN,Terraza vistas mar urbanización privada,NaN
2,3,Piso,Piso República de Colombia,Laredo,Zona Playa,600,1,58,9,1,1,0,0,0,0,0,1,Larga temporada terraza,NaN
3,4,Chalet independiente,Barrio de la Molina,Comillas,NaN,5000,6,392,NaN,NaN,NaN,1,0,1,1,0,NaN,Parcela 804 m2,NaN
4,5,Estudio,Espíritu Santo,Laredo,Centro,900,0,37,2,0,0,0,0,1,0,0,NaN,Vacacional sin ascensor,NaN


In [ ]:
df = df.iloc[:, :-1]


In [ ]:
df[["precio_mensual_eur", "habitaciones", "superficie_m2"]].dtypes


,0
precio_mensual_eur,int64
habitaciones,int64
superficie_m2,int64


In [ ]:
cols_bool = ["exterior", "ascensor", "garaje_incluido", "garaje_opcional", "temporada", "lujo", "vistas_mar", "amueblado"]

df[cols_bool] = (
    df[cols_bool]
        .astype(str)
        .apply(lambda x: x.str.strip().str.lower())
        .replace({
            "sí": 1,
            "si": 1,
            "no": 0,
            "1": 1,
            "true": 1,
            "false": 0,
            "0": 0,
            "null": np.nan,
            "nan": np.nan
        })
        .apply(pd.to_numeric, errors="coerce")
)


/tmp/ipython-input-415787893.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({


In [ ]:
df["amueblado"].value_counts(dropna=False)


,count
amueblado,
NaN,275
1.0,151
0.0,89


In [ ]:
df["planta"].value_counts(dropna=False)


,count
planta,
NaN,91
2,63
3,53
1,52
1ª,47
5,30
4,30
Bajo,30
2ª,25


In [ ]:
df["planta"].unique()


array(['1', '3', '9', nan, '2', '4', '7', '0', '5', '6', '8',
       'entreplanta', '14', '15', '16', '13', 'ático', '10', '12', '3ª',
       '5ª', '2ª', 'Bajo', '1ª', 'Entreplanta', '4ª', '9ª', '6ª', '13ª',
       '8ª', '14ª'], dtype=object)

In [ ]:
df["planta"] = (
    df["planta"]
        .astype(str)
        .str.lower()
        .str.strip()
)

df["planta"] = df["planta"].str.replace("ª", "", regex=False)

df["planta"] = df["planta"].replace({
    "bajo": 0,
    "entreplanta": 0.5,
    "atico": np.nan  # lo tratamos aparte
})

df["planta"] = pd.to_numeric(df["planta"], errors="coerce")

df.loc[df["planta"] > 20, "planta"] = np.nan

df.loc[df["tipo_inmueble"] == "casa", "planta"] = np.nan


In [ ]:
df["planta"].unique()


array([ 1. ,  3. ,  9. ,  nan,  2. ,  4. ,  7. ,  0. ,  5. ,  6. ,  8. ,
        0.5, 14. , 15. , 16. , 13. , 10. , 12. ])

In [ ]:
df["planta"].describe()


,planta
count,423.000000
mean,2.741135
std,2.537120
min,0.000000
25%,1.000000
50%,2.000000
75%,4.000000
max,16.000000


In [ ]:
df["tipo_inmueble"].unique()


array(['Piso', 'Chalet independiente', 'Estudio', 'Chalet pareado',
       'Ático', 'Dúplex', 'Chalet adosado', 'Casa independiente',
       'Chalet', 'Finca rústica', 'Caserón', 'Casa rural',
       'Casa de pueblo', 'Casa o chalet independiente'], dtype=object)

In [ ]:

# 0) Normalizar texto (muy importante)
df["tipo_inmueble"] = (
    df["tipo_inmueble"]
      .astype(str)
      .str.strip()
      .str.lower()
)

# 1) Crear 'subtipo' como copia del tipo original (antes de colapsar)
df["subtipo"] = df["tipo_inmueble"].replace({"nan": np.nan})

# 2) Definir qué cuenta como piso vs casa
set_piso = {"piso", "estudio", "ático", "atico", "dúplex", "duplex"}

# 3) Colapsar 'tipo_inmueble' a solo 'piso' o 'casa'
df["tipo_inmueble"] = np.where(df["tipo_inmueble"].isin(set_piso), "piso", "casa")

# 4) Estandarizar 'subtipo' (opcional: homogenizar tildes)
df["subtipo"] = df["subtipo"].replace({
    "atico": "ático",
    "duplex": "dúplex"
})

# 5) Rellenar subtipo según reglas:
# - Si es piso y subtipo está vacío o era "piso" -> "apartamento"
mask_piso = df["tipo_inmueble"].eq("piso")
mask_subtipo_vacio = df["subtipo"].isna() | df["subtipo"].eq("piso")

df.loc[mask_piso & mask_subtipo_vacio, "subtipo"] = "apartamento"

# - Si es casa y subtipo está vacío -> "casa independiente"
mask_casa = df["tipo_inmueble"].eq("casa")
df.loc[mask_casa & df["subtipo"].isna(), "subtipo"] = "casa independiente"


In [ ]:
valores_eliminar = [
"Finca rústica",
"finca rústica"
]

df = df[~df["subtipo"].isin(valores_eliminar)]

In [ ]:
df["subtipo"] = (
    df["subtipo"]
        .str.lower()
        .str.strip()
        .replace({
            "chalét independiente": "casa independiente",
            "chalet independiente": "casa independiente",
            "chalét adosado": "casa adosada",
            "chalet adosado": "casa adosada",
            "chalét pareado": "casa adosada",
            "chalet pareado": "casa adosada",
            "casa o chalet independiente": "casa independiente",
            "chalet": "casa independiente",
            })
)


In [ ]:
df["tipo_inmueble"].value_counts(dropna=False)


,count
tipo_inmueble,
piso,428
casa,86


In [ ]:
df["subtipo"].value_counts(dropna=False)


,count
subtipo,
apartamento,370
casa independiente,43
casa adosada,32
ático,28
dúplex,16
estudio,14
casa rural,7
casa de pueblo,3
caserón,1


In [ ]:
df = pd.get_dummies(
    df,
    columns=["tipo_inmueble", "subtipo"],
    drop_first=True
)

df.columns

Index(['id', 'titulo', 'municipio', 'zona', 'precio_mensual_eur',
       'habitaciones', 'superficie_m2', 'planta', 'exterior', 'ascensor',
       'garaje_incluido', 'garaje_opcional', 'temporada', 'lujo', 'vistas_mar',
       'amueblado', 'observaciones', 'tipo_inmueble_piso',
       'subtipo_casa adosada', 'subtipo_casa de pueblo',
       'subtipo_casa independiente', 'subtipo_casa rural', 'subtipo_caserón',
       'subtipo_dúplex', 'subtipo_estudio', 'subtipo_ático'],
      dtype='object')

In [ ]:
df["observaciones"] = (
    df["observaciones"]
        .astype(str)
        .str.lower()
        .str.strip()
)


patron_verano = r"\b(quincena|quincenas|verano|julio|agosto)\b"
patron_junio = r"\bjunio\b"
patron_excluir = r"(sep|sept|septiembre|hasta.*junio|temporal\s+sep|contrato temporal|larga duraci[oó]n)"

cond_verano_fuerte = df["observaciones"].str.contains(patron_verano, regex=True, na=False)
cond_junio = df["observaciones"].str.contains(patron_junio, regex=True, na=False)
cond_excluir = df["observaciones"].str.contains(patron_excluir, regex=True, na=False)

df["es_vacacional_verano"] = np.where(
    (cond_verano_fuerte | cond_junio) & (~cond_excluir),
    1,
    0
)


/tmp/ipython-input-2601730326.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  cond_verano_fuerte = df["observaciones"].str.contains(patron_verano, regex=True, na=False)
/tmp/ipython-input-2601730326.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  cond_excluir = df["observaciones"].str.contains(patron_excluir, regex=True, na=False)


In [ ]:
df[df["es_vacacional_verano"] == 1][["observaciones"]].head(20)


,observaciones
16,quincenas julio-agosto
22,no agosto parcial
25,agosto 3000 quincena
71,temporada julio/agosto lujo
76,temporada quincenas julio
80,verano quincenas rebaja 1900->1700 vistas mar
81,verano quincenas rebaja 1200->1100
82,verano reformado amueblado sin ascensor
83,julio 3300 agosto 3800 (temporal)
85,verano quincenas


In [ ]:
df[df["observaciones"].str.contains("verano|julio|agosto", na=False)][["observaciones"]]


,observaciones
16,quincenas julio-agosto
22,no agosto parcial
25,agosto 3000 quincena
71,temporada julio/agosto lujo
76,temporada quincenas julio
80,verano quincenas rebaja 1900->1700 vistas mar
81,verano quincenas rebaja 1200->1100
82,verano reformado amueblado sin ascensor
83,julio 3300 agosto 3800 (temporal)
85,verano quincenas


In [ ]:
df.head(20)


,id,titulo,municipio,zona,precio_mensual_eur,habitaciones,superficie_m2,planta,exterior,ascensor,...,tipo_inmueble_piso,subtipo_casa adosada,subtipo_casa de pueblo,subtipo_casa independiente,subtipo_casa rural,subtipo_caserón,subtipo_dúplex,subtipo_estudio,subtipo_ático,es_vacacional_verano
0,1,Piso en los Pinares,Santander,Los Castros,850,3,63,1.0,1.0,1.0,...,True,False,False,False,False,False,False,False,False,0
1,2,Piso Calleja Norte,Santander,El Sardinero,1400,3,107,3.0,1.0,1.0,...,True,False,False,False,False,False,False,False,False,0
2,3,Piso República de Colombia,Laredo,Zona Playa,600,1,58,9.0,1.0,1.0,...,True,False,False,False,False,False,False,False,False,0
3,4,Barrio de la Molina,Comillas,NaN,5000,6,392,NaN,NaN,NaN,...,False,False,False,True,False,False,False,False,False,0
4,5,Espíritu Santo,Laredo,Centro,900,0,37,2.0,0.0,0.0,...,True,False,False,False,False,False,False,True,False,0
5,6,Polanco,Polanco,NaN,1750,4,150,NaN,NaN,NaN,...,False,True,False,False,False,False,False,False,False,0
6,7,Puerto Chico,Santander,Puerto Chico,575,1,50,4.0,NaN,NaN,...,True,False,False,False,False,False,False,False,True,0
7,8,Luis Buñuel,Santander,Peñacastillo,1300,3,150,NaN,NaN,NaN,...,False,True,False,False,False,False,False,False,False,0
8,9,San Andrés,Castro-Urdiales,Cotolino,625,1,47,1.0,1.0,1.0,...,True,False,False,False,False,False,False,False,False,0
9,10,Cuatro Caminos,Santander,Cuatro Caminos,800,2,90,2.0,1.0,0.0,...,True,False,False,False,False,False,False,False,False,0
